<a href="https://colab.research.google.com/github/smsag99/Thesis/blob/main/codes/Hierchical_Structure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMPORTANT :



In [1]:
path = '/content/drive/MyDrive/Thesis_Data/'

## preprocessing Data (removing outliers)

In [4]:
def remove_outliers(df_animal):
  df_animal_filtered = df_animal[(df_animal['milk_kg'] >= 0.4) & (df_animal['milk_kg'] <= 26.5)]
  print(f"Total rows removed after milk filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered = df_animal_filtered[(df_animal_filtered['fat_p'] >= 1.6) & (df_animal_filtered['fat_p'] <= 15.05)]
  print(f"Total rows removed after fat filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered = df_animal_filtered[(df_animal_filtered['protein_p'] >= 2.67) & (df_animal_filtered['protein_p'] <= 6.68)]
  print(f"Total rows removed after protein filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered = df_animal_filtered[(df_animal_filtered['SCS'] >= -2.06) & (df_animal_filtered['SCS'] <= 10.73)]
  print(f"Total rows removed after SCS filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered = df_animal_filtered[df_animal_filtered['ECM'] <= 301]
  print(f"Total rows removed after ECM filtering: {len(df_animal)-len(df_animal_filtered)}")
  df_animal_filtered['AFC'] = pd.to_timedelta(df_animal_filtered['AFC'])
  df_animal_filtered = df_animal_filtered[(df_animal_filtered['AFC'] >= pd.Timedelta(days=720)) & (df_animal_filtered['AFC'] <= pd.Timedelta(days=1440))]
  print(f"Total rows removed after AFC filtering: {len(df_animal)-len(df_animal_filtered)}")
  print(f"Total rows after filtering: {len(df_animal_filtered)}")
  return df_animal_filtered

df_Merged_Scaled_Processed = remove_outliers(df_Merged_Scaled_uprocessed)
df_Merged_Scaled_Processed.to_csv(path + '/Final_Data/Merged_Data_Scaled_Processed.csv', index=False)
print("saved")

NameError: name 'df_Merged_Scaled_uprocessed' is not defined

In [2]:
def convert_types(df):
  df['dtb'] = pd.to_datetime(df['dtb'])
  df['dtt'] = pd.to_datetime(df['dtt'])
  df['dtc'] = pd.to_datetime(df['dtc'])
  df['AFC'] = pd.to_timedelta(df['AFC']).dt.days
  df['DIM'] = pd.to_timedelta(df['DIM']).dt.days
  return df

## Claude Method

### Step 1 — Build the summing matrix S
#### This is the core of HTS. The S matrix maps bottom-level series (Animal × Parity) to every level above.


In [29]:
import pandas as pd
import numpy as np

df = pd.read_csv(path + '/Final_Data/Merged_Data_Scaled_Processed.csv')
df = convert_types(df)
df['dtt'] = pd.to_datetime(df['dtt'])

# Create a unique bottom-level key: Animal_Parity
df['series_key'] = df['Animal_ID'] + '_P' + df['parity'].astype(str)

# Pivot: each column = one bottom-level series, rows = dates
bottom = df.pivot_table(index='dtt', columns='series_key', values='milk_kg', aggfunc='sum').fillna(0)

# Animal-level aggregates (sum parities per animal per date)
animal_level = df.pivot_table(index='dtt', columns='Animal_ID', values='milk_kg', aggfunc='sum').fillna(0)

# Farm-level aggregate
farm_level = df.pivot_table(index='dtt', columns='Farm_Code', values='milk_kg', aggfunc='sum').fillna(0)

### Step 2 — Use the hts or statsforecast library
#### The best Python option today is hierarchicalforecast from Nixtla:

In [31]:
%%capture
pip install hierarchicalforecast statsforecast

In [32]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

# Define hierarchy as a list-of-lists: [farm, animal, series_key]
# Each row of df needs all three levels filled
df['Farm'] = df['Farm_Code'].astype(str)
df['Animal'] = df['Animal_ID']
df['Parity'] = 'P' + df['parity'].astype(str)

# Build the Y_df in Nixtla format: unique_id, ds, y
Y_df = df[['series_key', 'dtt', 'milk_kg']].rename(columns={
    'series_key': 'unique_id', 'dtt': 'ds', 'milk_kg': 'y'
})

# Also add upper-level series manually
animal_df = df.groupby(['Animal_ID', 'dtt'])['milk_kg'].sum().reset_index()
animal_df['unique_id'] = animal_df['Animal_ID']
animal_df = animal_df.rename(columns={'dtt': 'ds', 'milk_kg': 'y'})

farm_df = df.groupby(['Farm_Code', 'dtt'])['milk_kg'].sum().reset_index()
farm_df['unique_id'] = farm_df['Farm_Code'].astype(str)
farm_df = farm_df.rename(columns={'dtt': 'ds', 'milk_kg': 'y'})

# Combine all levels
Y_df_all = pd.concat([
    farm_df[['unique_id','ds','y']],
    animal_df[['unique_id','ds','y']],
    Y_df
]).sort_values(['unique_id', 'ds'])

### Step 3 — Define the hierarchy tags

In [33]:
# S_df: summing matrix (columns = bottom series, rows = all series)
# tags: dict mapping each level name to the list of series at that level

tags = {
    'Farm':   farm_df['unique_id'].unique().tolist(),
    'Animal': animal_df['unique_id'].unique().tolist(),
    'Parity': df['series_key'].unique().tolist()
}

### Step 4 — Fit base forecasters and reconcile

In [34]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

# Fit one model per bottom-level series
sf = StatsForecast(models=[AutoETS(season_length=12)], freq='MS', n_jobs=-1)
sf.fit(Y_df_all)
forecasts_df = sf.predict(h=6)  # 6 periods ahead

# Reconcile: MinTrace is the gold standard
hrec = HierarchicalReconciliation(reconcilers=[
    BottomUp(),
    MinTrace(method='mint_shrink')
])

reconciled = hrec.reconcile(
    Y_hat_df=forecasts_df,
    Y_df=Y_df_all,
    tags=tags
)



KeyError: 'fitted'

## Gemini Method


### Step 1: Preprocess the Data

In [3]:
import pandas as pd

# 1. Load and parse dates
df = pd.read_csv(path + '/Final_Data/Merged_Data_Scaled_Processed.csv')
df = convert_types(df)
df['dtt'] = pd.to_datetime(df['dtt'])

# 2. Select relevant columns
ts_df = df[['Farm_Code', 'Animal_ID', 'dtt', 'milk_kg']].copy()

# 3. Create a unique identifier for the bottom level
ts_df['Animal_Key'] = ts_df['Farm_Code'].astype(str) + "_" + ts_df['Animal_ID'].astype(str)
ts_df = ts_df.rename(columns={'dtt': 'ds', 'milk_kg': 'y'})

# 4. Resample to a regular frequency (e.g., Monthly 'ME')
ts_df = ts_df.set_index('ds').groupby('Animal_Key').resample('ME')['y'].sum().reset_index()

In [4]:
# 5. SAVE TO CSV
ts_df.to_csv('preprocessed_hts_data.csv', index=False)
print("Step 1 complete. Data saved to preprocessed_hts_data.csv")

Step 1 complete. Data saved to preprocessed_hts_data.csv


### Step 2: Build the Hierarchy aggregation

In [1]:
%%capture
!pip install hierarchicalforecast statsforecast

In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix
import gc

# 1. Load data
ts_df = pd.read_csv('preprocessed_hts_data.csv', parse_dates=['ds'])
ts_df['Farm'] = ts_df['Animal_Key'].apply(lambda x: x.split('_')[0])

print("Aggregating Y_df...")
# Build levels manually
bottom_df = ts_df[['Animal_Key', 'ds', 'y']].rename(columns={'Animal_Key': 'unique_id'})
farm_df = ts_df.groupby(['Farm', 'ds'])['y'].sum().reset_index().rename(columns={'Farm': 'unique_id'})
total_df = ts_df.groupby('ds')['y'].sum().reset_index()
total_df['unique_id'] = 'Total'

Y_df = pd.concat([total_df, farm_df, bottom_df], ignore_index=True)

# Free up memory immediately
del ts_df, total_df, farm_df, bottom_df
gc.collect()

print("Extracting unique nodes...")
unique_farms = list(Y_df[Y_df['unique_id'].str.contains('_') == False]['unique_id'].unique())
unique_farms.remove('Total')
unique_animals = list(Y_df[Y_df['unique_id'].str.contains('_')]['unique_id'].unique())

tags = {
    'Farm': np.array(unique_farms),
    'Farm/Animal_Key': np.array(unique_animals)
}

print("Building Sparse S_df matrix...")
# 2. Build S_df as a Sparse Matrix
all_nodes = ['Total'] + unique_farms + unique_animals
node_to_row = {node: i for i, node in enumerate(all_nodes)}

rows = []
cols = []
vals = []

# Populate the sparse coordinates
for col_idx, animal in enumerate(unique_animals):
    farm = animal.split('_')[0]

    # Every animal column has exactly three '1's:
    # 1. At the Total row
    rows.append(node_to_row['Total'])
    cols.append(col_idx)
    vals.append(1)

    # 2. At its specific Farm row
    rows.append(node_to_row[farm])
    cols.append(col_idx)
    vals.append(1)

    # 3. At its own Animal row
    rows.append(node_to_row[animal])
    cols.append(col_idx)
    vals.append(1)

# Create the SciPy sparse matrix (COO format is extremely fast for building)
# Using uint8 to keep it as tiny as possible
S_sparse = coo_matrix((vals, (rows, cols)), shape=(len(all_nodes), len(unique_animals)), dtype=np.uint8)

# Convert directly to a Pandas DataFrame with Sparse data types
S_df = pd.DataFrame.sparse.from_spmatrix(S_sparse, index=all_nodes, columns=unique_animals)

print("Step 2 completed successfully! RAM usage optimized.")

Aggregating Y_df...
Extracting unique nodes...
Building Sparse S_df matrix...
Step 2 completed successfully! RAM usage optimized.


### Step 3: Train Base Models

In [ ]:
%%capture


In [13]:
# Generate base forecasts
sf = StatsForecast(models=models, freq='M', n_jobs=-1)

# FIT AND PREDICT WITH 'fitted=True'
Y_hat_df = sf.forecast(df=Y_df_filtered, h=6, fitted=True)

# Extract the in-sample fitted values
Y_fitted_df = sf.forecast_fitted_values()
# Drop the duplicate 'y' column so we don't get overlapping names
Y_fitted_df = Y_fitted_df.drop(columns=['y'], errors='ignore')

# Merge the predictions into your historical dataframe
Y_df_filtered = Y_df_filtered.merge(Y_fitted_df, on=['unique_id', 'ds'], how='left')

print("Fitted values added! Ready for MinTrace.")

Fitted values added! Ready for MinTrace.


/usr/local/lib/python3.12/dist-packages/utilsforecast/processing.py:378: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  freq = pd.tseries.frequencies.to_offset(freq)
/usr/local/lib/python3.12/dist-packages/utilsforecast/processing.py:434: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  freq = pd.tseries.frequencies.to_offset(freq)


### Step 4: Reconcile the Forecasts

In [14]:
import numpy as np
from hierarchicalforecast.methods import BottomUp, MinTrace
from hierarchicalforecast.core import HierarchicalReconciliation

# --- Align S_df and tags with the filtered Y_hat_df ---
valid_ids = set(Y_hat_df['unique_id'].unique())

# 1. Filter rows in S_df
if 'unique_id' not in S_df.columns:
    S_df = S_df.reset_index(names='unique_id')
S_df_filtered = S_df[S_df['unique_id'].isin(valid_ids)].copy()

# 2. Filter columns in S_df
valid_columns = [col for col in S_df_filtered.columns if col == 'unique_id' or col in valid_ids]
S_df_filtered = S_df_filtered[valid_columns]

# 3. Filter the tags dictionary
tags_filtered = {}
for level_name, nodes in tags.items():
    valid_nodes_for_level = [node for node in nodes if node in valid_ids]
    if len(valid_nodes_for_level) > 0:
        tags_filtered[level_name] = np.array(valid_nodes_for_level)

print(f"S_df shape reduced to {S_df_filtered.shape}")
# -----------------------------------------------------------

# Change this inside your reconcilers list in Step 4
reconcilers = [
    BottomUp(),
    MinTrace(method='wls_struct')  # Uses structural weights instead of residuals
]

# Initialize the reconciler
hforecast = HierarchicalReconciliation(reconcilers=reconcilers)

# Apply reconciliation
Y_rec_df = hforecast.reconcile(
    Y_hat_df=Y_hat_df,
    Y_df=Y_df_filtered,
    S_df=S_df_filtered,
    tags=tags_filtered
)

print(Y_rec_df.head())

S_df shape reduced to (3848, 3570)


Exception: min_trace (wls_struct) is ill-conditioned. Please use another reconciliation method.